In [1]:
%env ANYWIDGET_HMR=1
from guidepost import Guidepost, Campsite

gp = Guidepost()
cs = Campsite()

env: ANYWIDGET_HMR=1


In [2]:
cs.test_server()

0


In [3]:
import pandas as pd
# jobs_data = pd.read_parquet("../data/test_data_med.parquet")
# jobs_data = pd.read_parquet("../data/kestrel_data_2024_01_28_subsample.parquet")
# jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-THETA_20240101_20241231.csv.gz", compression='gzip')
jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')

# jobs_data
gp.records = jobs_data
cs.records = jobs_data

/tmp/ipykernel_1241401/1879876493.py:5: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  jobs_data = pd.read_csv("../data/ANL-ALCF-DJC-POLARIS_20250101_20251231.csv.gz", compression='gzip')


In [4]:
cs

Campsite()

In [6]:
Im curious about what conditions cause job failures on this system. I want to use some kind of causal regression analysis and I believe that nodes with more usage tend cause more failures even when normaized for usage 

SyntaxError: invalid syntax (546510929.py, line 1)

In [6]:
!pip install seaborn/home/cscullyallison/Programming/guidepost/guidepost/static/trailmark.js.map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 14.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 23.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [seaborn]m7/8 [seaborn]ib]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [10]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# Optional: for richer regression-based summaries
try:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

def _welch_df(a, b):
    na, nb = len(a), len(b)
    s1, s2 = np.var(a, ddof=1), np.var(b, ddof=1)
    num = (s1/na + s2/nb) ** 2
    denom = (s1/na)**2/(na-1) + (s2/nb)**2/(nb-1)
    if denom == 0:
        return np.nan
    return num / denom

def bootstrap_mean_diff(x, y, n_boot=2000, random_state=None):
    rng = np.random.default_rng(random_state)
    x = np.asarray(x)
    y = np.asarray(y)
    diffs = []
    n = min(len(x), len(y))  # small safeguard
    for _ in range(n_boot):
        ix = rng.integers(0, len(x), len(x))
        iy = rng.integers(0, len(y), len(y))
        diffs.append(np.mean(x[ix]) - np.mean(y[iy]))
    lo = np.percentile(diffs, 2.5)
    hi = np.percentile(diffs, 97.5)
    return float(lo), float(hi)

def cohen_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    varx, vary = np.var(x, ddof=1), np.var(y, ddof=1)
    pooled = np.sqrt(((nx-1)*varx + (ny-1)*vary) / (nx + ny - 2))
    if pooled == 0:
        return 0.0
    return (np.mean(x) - np.mean(y)) / pooled

def hypothesis_test_walltime_overburn(df, walltime_col='WALLTIME_SECONDS', overburn_col='IS_OVERBURN',
                                    alpha=0.05, bootstrap_n=2000, random_state=42,
                                    plot_figs=True, save_plots=False, plots_dir='plots'):
    """
    Performs the hypothesis test:
      H0: μ(WALLTIME_SECONDS | IS_OVERBURN=0) = μ(WALLTIME_SECONDS | IS_OVERBURN=1)
      H1: μ(WALLTIME_SECONDS | IS_OVERBURN=0) != μ(WALLTIME_SECONDS | IS_OVERBURN=1)
    Also provides:
      - Welch's t-test on WALLTIME_SECONDS
      - Optional: Mann-Whitney U test
      - Optional: t-test on log(WALLTIME_SECONDS + 1)
      - Effect sizes (Cohen's d, rank-biserial for MWU if desired)
      - Bootstrap confidence interval for the mean difference
      - Basic visualization (distribution by group)
    Returns a dictionary with results.
    """
    x = df[walltime_col]
    g = df[overburn_col]

    # Drop missing
    mask = x.notna() & g.notna()
    x = x[mask]
    g = g[mask]

    x0 = x[g == 0]
    x1 = x[g == 1]

    n0, n1 = len(x0), len(x1)

    # Basic descriptive stats
    mean0, mean1 = float(np.mean(x0)), float(np.mean(x1))
    med0, med1 = float(np.median(x0)), float(np.median(x1))
    std0, std1 = float(np.std(x0, ddof=1)), float(np.std(x1, ddof=1))
    t_stats = {}

    # Welch's t-test on WALLTIME_SECONDS
    tstat, pval = stats.ttest_ind(x0, x1, equal_var=False, nan_policy='omit')
    df_welch = _welch_df(x0, x1)
    t_stats['welch_t'] = tstat
    t_stats['welch_p'] = pval
    t_stats['welch_df'] = df_welch

    # Effect size (Cohen's d) on WALLTIME_SECONDS
    d = cohen_d(x0, x1)

    # Bootstrap CI for mean difference
    lo_boot, hi_boot = bootstrap_mean_diff(x0, x1, n_boot=bootstrap_n, random_state=random_state)

    # Optional: Mann-Whitney U test (nonparametric)
    mw_stat, mw_p = stats.mannwhitneyu(x0, x1, alternative='two-sided')

    # Optional: log transform (handle skew)
    x0_log = np.log1p(x0)
    x1_log = np.log1p(x1)
    tlog, pval_log = stats.ttest_ind(x0_log, x1_log, equal_var=False, nan_policy='omit')
    # Cohen's d for log-scale (interpret carefully)
    d_log = cohen_d(x0_log, x1_log)

    results = {
        'n0': int(n0),
        'n1': int(n1),
        'mean0': float(mean0),
        'mean1': float(mean1),
        'sd0': float(std0),
        'sd1': float(std1),
        'median0': float(med0),
        'median1': float(med1),
        'welch_df': float(df_welch) if not np.isnan(df_welch) else np.nan,
        'welch_t': float(tstat),
        'welch_p': float(pval),
        'cum_effect_size_d': float(d),
        'bootstrap_ci_lo': float(lo_boot),
        'bootstrap_ci_hi': float(hi_boot),
        'mw_u': float(mw_stat),
        'mw_p': float(mw_p),
        'log_t': float(tlog),
        'log_p': float(pval_log),
        'log_cohen_d': float(d_log),
    }

    # Pretty print of results
    print("Hypothesis test results (WALLTIME_SECONDS by IS_OVERBURN):")
    print("Group 0 (IS_OVERBURN=0): n={}, mean={:.3f}, sd={:.3f}, median={:.3f}".format(n0, mean0, std0, med0))
    print("Group 1 (IS_OVERBURN=1): n={}, mean={:.3f}, sd={:.3f}, median={:.3f}".format(n1, mean1, std1, med1))
    print("Welch's t-test: t = {:.4f}, df = {:.2f}, p = {:.4g}".format(results['welch_t'], results['welch_df'], results['welch_p']))
    print("Effect size (Cohen's d): {:.4f}".format(results['cum_effect_size_d']))
    print("Bootstrap 95% CI for mean difference (group0 - group1): [{:.3f}, {:.3f}]".format(lo_boot, hi_boot))
    print("Mann-Whitney U test: U = {:.3f}, p = {:.4g}".format(mw_stat, mw_p))
    print("Log-WELCH t-test on log(WALLTIME_SECONDS+1): t = {:.4f}, p = {:.4g}".format(results['log_t'], results['log_p']))
    print("Log-scale Cohen's d: {:.4f}".format(results['log_cohen_d']))

    if plot_figs:
        # Distribution plots
        plt.figure(figsize=(10, 4))
        sns.kdeplot(x0, label='IS_OVERBURN=0', shade=True)
        sns.kdeplot(x1, label='IS_OVERBURN=1', shade=True)
        plt.title('WALLTIME_SECONDS distribution by IS_OVERBURN')
        plt.xlabel('WALLTIME_SECONDS')
        plt.legend()
        if save_plots:
            plt.savefig(f"{plots_dir}/walltime_by_overburn_kde.png", dpi=300, bbox_inches='tight')
        plt.show()

        # Box/violin plot
        plt.figure(figsize=(6, 4))
        data_plot = df[[overburn_col, walltime_col]].dropna()
        sns.boxplot(x=overburn_col, y=walltime_col, data=data_plot)
        plt.title('WALLTIME_SECONDS by IS_OVERBURN')
        plt.xlabel('IS_OVERBURN')
        plt.ylabel('WALLTIME_SECONDS')
        if save_plots:
            plt.savefig(f"{plots_dir}/walltime_by_overburn_box.png", dpi=300, bbox_inches='tight')
        plt.show()

    return results

def summarize_features(df, max_cols=None):
    """
    Produce a high-level overview of all features:
    - For numeric columns: n, n_missing, mean, std, min, 25/50/75 percentiles, max, n_unique
    - For categorical/object columns: n_unique, top, top_freq, approximate missing
    Returns a dictionary with per-column summaries, and prints a readable summary.
    """
    if max_cols is not None and max_cols < len(df.columns):
        cols = df.columns[:max_cols]
        cols = list(cols)
    else:
        cols = list(df.columns)

    overview = {}
    total_rows = len(df)
    for col in cols:
        ser = df[col]
        ser_nonnull = ser.dropna()
        n_missing = int(ser.isna().sum())
        n = len(ser)
        if pd.api.types.is_numeric_dtype(ser):
            col_summary = {
                'dtype': 'numeric',
                'n': n,
                'n_missing': n_missing,
                'mean': float(ser_nonnull.mean()) if len(ser_nonnull) else np.nan,
                'std': float(ser_nonnull.std(ddof=1)) if len(ser_nonnull) > 1 else np.nan,
                'min': float(ser_nonnull.min()) if len(ser_nonnull) else np.nan,
                '25%': float(ser_nonnull.quantile(0.25)) if len(ser_nonnull) else np.nan,
                '50%': float(ser_nonnull.median()) if len(ser_nonnull) else np.nan,
                '75%': float(ser_nonnull.quantile(0.75)) if len(ser_nonnull) else np.nan,
                'max': float(ser_nonnull.max()) if len(ser_nonnull) else np.nan,
                'n_unique': int(ser.nunique())
            }
        else:
            # Categorical/object
            top_val = ser_nonnull.value_counts().idxmax() if len(ser_nonnull) > 0 else np.nan
            top_freq = int(ser_nonnull.value_counts().max()) if len(ser_nonnull) > 0 else 0
            col_summary = {
                'dtype': 'categorical',
                'n': n,
                'n_missing': n_missing,
                'n_unique': int(ser.nunique()),
                'top': top_val,
                'top_freq': top_freq
            }
        overview[col] = col_summary

    # Print a compact, readable overview
    print("Feature overview (high-level):")
    for col, s in overview.items():
        if s['dtype'] == 'numeric':
            print("  - {col}: numeric | n={n}, missing={missing}, mean={mean:.3g}, std={std:.3g}, min={min:.3g}, 25%={q25:.3g}, 50%={q50:.3g}, 75%={q75:.3g}, max={max:.3g}, n_unique={nu}".format(
                col=col, n=s['n'], missing=s['n_missing'], mean=s['mean'], std=s['std'],
                min=s['min'], q25=s['25%'], q50=s['50%'], q75=s['75%'], max=s['max'],
                nu=s['n_unique']))
        else:
            print("  - {col}: categorical | n={n}, missing={missing}, n_unique={nu}, top={top} ({top_freq}x)".format(
                col=col, n=s['n'], missing=s['n_missing'], nu=s['n_unique'], top=s['top'], top_freq=s['top_freq']))
    print("Total columns analyzed: {}".format(len(overview)))
    return overview

# summarize_features(jobs_data)
results = hypothesis_test_walltime_overburn(
    jobs_data,
    walltime_col='WALLTIME_SECONDS',
    overburn_col='IS_OVERBURN',
    alpha=0.05,
    bootstrap_n=2000,
    random_state=42,
    plot_figs=True,
    save_plots=False
)
    
    # Example usage (uncomment and adapt to your environment):
# if __name__ == "__main__":
#     # Load your data
#     df = pd.read_csv("your_data.csv")  # ensure WALLTIME_SECONDS and IS_OVERBURN exist
#
#     # 1) Hypothesis test
#     results = hypothesis_test_walltime_overburn(
#         df,
#         walltime_col='WALLTIME_SECONDS',
#         overburn_col='IS_OVERBURN',
#         alpha=0.05,
#         bootstrap_n=2000,
#         random_state=42,
#         plot_figs=True,
#         save_plots=False
#     )
#
#     # 2) Feature overview
#     overview = summarize_features(df)

In [ ]:
jobs_data.columns

In [ ]:
gp.vis_configs = {
        'x': 'START_TIMESTAMP',
        'y': 'USED_CORE_HOURS',
        'color': 'processors_req',
        'color_agg': 'avg',
        'categorical': 'user',
        'facet_by': 'MACHINE_PARTITION'
}

In [ ]:
gp

In [ ]:
I am interested in relationships between memory efficiency, power, and job type.

In [ ]:
gp.selection

In [ ]:
import pandas as pd

# Split data into long-running and shorter jobs

long = jobs_data[jobs_data['wallclock_req_seconds'] > 14400]

short = jobs_data[jobs_data['wallclock_req_seconds'] <= 14400]

# Compute average CPU efficiency for each group

avg_cpu_eff_long = long['cpu_eff'].mean()

avg_cpu_eff_short = short['cpu_eff'].mean()

# Hypothesis: avg CPU efficiency for long < avg CPU efficiency for short

result = avg_cpu_eff_long < avg_cpu_eff_short

In [ ]:
print(result, avg_cpu_eff_long, avg_cpu_eff_short)

In [ ]:
import pandas as pd

def evaluate(df):

    avg_standard = df.loc[df['partition'] == 'standard', 'wallclock_req_seconds'].mean()
    
    avg_short = df.loc[df['partition'] == 'short', 'wallclock_req_seconds'].mean()

    print(avg_standard, avg_short)
    
    return avg_standard > avg_short

evaluate(jobs_data)

In [ ]:
import pandas as pd

def evaluate(df):

    corr = df['cpu_eff'].corr(df['wallclock_req_seconds'])

    result = corr < -0.5

    print(corr)

    return result

In [ ]:
evaluate_hypothesis(jobs_data)

In [ ]:
import pandas as pd

def evaluate(df):

    # Long-running jobs (wallclock_req_seconds > 3600)
    
    long_running = df[df['wallclock_req_seconds'] > 3600]
    
    avg_long = long_running['cpu_eff'].mean()
    
    # Average CPU efficiency across all jobs
    
    avg_all = df['cpu_eff'].mean()

    print(avg_long, avg_all)
    
    return avg_long > avg_all

In [ ]:
def evaluate_hypothesis(df):

    # Compute failure rate for account 'sipv'

    total_sipv = df[df['account'] == 'sipv'].shape[0]

    failed_sipv = df[(df['account'] == 'sipv') & (df['state'] == 'FAILED')].shape[0]

    failure_rate_sipv = failed_sipv / total_sipv if total_sipv > 0 else float('nan')

    # Compute failure rate for all other accounts

    total_others = df[df['account'] != 'sipv'].shape[0]

    failed_others = df[(df['account'] != 'sipv') & (df['state'] == 'FAILED')].shape[0]

    failure_rate_others = failed_others / total_others if total_others > 0 else float('nan')

    # If we don't have data for either group, return False (cannot conclude higher rate)

    if total_sipv == 0 or total_others == 0:

        return False

    print(failure_rate_sipv, failure_rate_others)

    return bool(failure_rate_sipv > failure_rate_others)